# Session 4: Agent Skills & MCP — Composable Capabilities for Deep Agents

**Data Science Dojo x SambaNova | Deep Agents Webinar Series**

In Session 3 we mastered context engineering — compression, isolation, and `create_deep_agent()` as our package base. Today we make agents **capable** by packaging reusable behaviors as **Agent Skills** (`SKILL.md`) and connecting to the outside world via **Model Context Protocol (MCP)**.

## A note on origins

**Agent Skills were created by Anthropic** (Oct 16, 2025) and the **open spec was published Dec 18, 2025**. Since then, OpenAI, Google, GitHub, Cursor, and the `deepagents` (LangChain) library have all adopted the same `SKILL.md` pattern. This is now a **framework-agnostic, open standard** — not vendor lock-in. The same skill folder runs unchanged across every major agent platform. Today we use `deepagents` because of its native `SkillsMiddleware`, but everything you author here is portable.

## What we'll cover

1. **Anatomy of a Skill** — `SKILL.md` frontmatter, body, bundled files
2. **Progressive disclosure** — the 3-tier loading pattern that makes 50 skills cost <1K tokens
3. **Ground-up vs library** — hand-roll a SKILL.md loader, then switch to `skills=[...]` in one line
4. **Demo 1**: Drop in Anthropic's published `pdf` skill
5. **Demo 2**: Author a custom `slide-deck` skill — ~30 lines of markdown, no code
6. **Demo 3**: Compose 3 skills + a subagent into a deep-research agent
7. **MCP × Skills** — how the two specs combine in production

## Section 0: Setup

In [31]:
import os
from dotenv import load_dotenv, find_dotenv

# Walk up parents to find the repo root .env
load_dotenv(find_dotenv(), override=True)
print('SAMBANOVA key:', 'set' if os.environ.get('SAMBANOVA_API_KEY') else 'MISSING')
print('Tavily key:', 'set' if os.environ.get('TAVILY_API_KEY') else 'MISSING')

SAMBANOVA key: set
Tavily key: set


In [32]:
import sys
sys.path.insert(0, os.path.join(".."))
from utils import format_messages, show_prompt

from rich.console import Console
from rich.markdown import Markdown
from rich.syntax import Syntax
from rich.panel import Panel

_console = Console()
console = _console  # alias so existing rich.console.print(...) usages keep working

def pretty_md(text: str, title: str | None = None) -> None:
    """Render markdown with rich so the audience sees real headings, lists, code blocks."""
    md = Markdown(text)
    if title:
        _console.print(Panel(md, title=title, border_style="magenta", padding=(1, 2)))
    else:
        _console.print(md)

def pretty_skill(path: str) -> None:
    """Show a SKILL.md as syntax-highlighted YAML frontmatter + rendered markdown body."""
    raw = open(path).read()
    import re
    m = re.match(r"^---\n(.*?)\n---\n(.*)", raw, re.DOTALL)
    if not m:
        _console.print(Markdown(raw)); return
    fm, body = m.group(1), m.group(2)
    _console.print(Panel(Syntax(fm, "yaml", theme="monokai", word_wrap=True),
                         title=f"📄 {path} — frontmatter", border_style="magenta"))
    _console.print(Panel(Markdown(body[:1500] + ("\n\n…[truncated]" if len(body) > 1500 else "")),
                         title=f"📄 {path} — body", border_style="cyan", padding=(1, 2)))

print("Helpers ready: format_messages, show_prompt, pretty_md, pretty_skill")


Helpers ready: format_messages, show_prompt, pretty_md, pretty_skill


In [33]:
# LangSmith tracing is automatic when env vars are set
if os.environ.get("LANGSMITH_API_KEY") and os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    print(f"LangSmith tracing enabled — project: {os.environ.get('LANGSMITH_PROJECT', 'default')}")
else:
    print("LangSmith not configured — tracing disabled.")

LangSmith tracing enabled — project: deep-agents-from-scratch


In [34]:
from langchain_sambanova import ChatSambaNova
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend

MODEL = ChatSambaNova(model="MiniMax-M2.7", temperature=0.0, max_tokens=16000)

# Skills live on disk at ./skills/. FilesystemBackend mounts the current
# directory as the agent's virtual filesystem root ('/'), so the agent
# can read /skills/pdf/SKILL.md etc. without us copying files into state.
BACKEND = FilesystemBackend(root_dir=".", virtual_mode=True)
SKILLS_DIR = "./skills"
print(f"Model: MiniMax-M2.7 | Backend: FilesystemBackend(.)")
print(f"Skills on disk: {sorted(d for d in os.listdir(SKILLS_DIR) if os.path.isdir(os.path.join(SKILLS_DIR, d)))}")


Model: MiniMax-M2.7 | Backend: FilesystemBackend(.)
Skills on disk: ['daily-standup', 'pdf', 'release-notes', 'skill-creator', 'slide-deck', 'web-research', 'xlsx']


---
## Section 1: Anatomy of an Agent Skill

A Skill is a **directory** with a `SKILL.md` entrypoint. The `SKILL.md` has:

1. **YAML frontmatter** — `name`, `description`, and optional fields like `allowed-tools`
2. **Markdown body** — the Standard Operating Procedure the agent follows
3. **Bundled files** — reference markdown and scripts loaded only on demand

Let's read Anthropic's `pdf` skill and see its anatomy directly.

In [35]:
import re
from pathlib import Path

def parse_skill_md(path: str) -> dict:
    """Tiny SKILL.md parser: split YAML frontmatter from markdown body."""
    raw = Path(path).read_text()
    m = re.match(r"^---\n(.*?)\n---\n(.*)", raw, re.DOTALL)
    if not m:
        return {"frontmatter": {}, "body": raw}
    fm_text, body = m.group(1), m.group(2)
    import yaml
    fm = yaml.safe_load(fm_text) or {}
    return {"frontmatter": fm, "body": body}

# Pretty-render the pdf skill — frontmatter as YAML, body as Markdown
pretty_skill("./skills/pdf/SKILL.md")

# Also keep the parsed dict for size analysis below
pdf_skill = parse_skill_md("./skills/pdf/SKILL.md")
print(f"\nFrontmatter keys: {list(pdf_skill['frontmatter'].keys())}")
print(f"Body length: {len(pdf_skill['body']):,} chars (~{len(pdf_skill['body'])//4:,} tokens)")


╭──────────────────────────────────── 📄 ./skills/pdf/SKILL.md — frontmatter ─────────────────────────────────────╮
│ name: pdf                                                                                                       │
│ description: Use this skill whenever the user wants to do anything with PDF files. This includes reading or     │
│ extracting text/tables from PDFs, combining or merging multiple PDFs into one, splitting PDFs apart, rotating   │
│ pages, adding watermarks, creating new PDFs, filling PDF forms, encrypting/decrypting PDFs, extracting images,  │
│ and OCR on scanned PDFs to make them searchable. If the user mentions a .pdf file or asks to produce one, use   │
│ this skill.                                                                                                     │
│ license: Proprietary. LICENSE.txt has complete terms                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────── 📄 ./skills/pdf/SKILL.md — body ────────────────────────────────────────╮
│                                                                                                                 │
│                                              PDF Processing Guide                                               │
│                                                                                                                 │
│  Overview                                                                                                       │
│                                                                                                                 │
│  This guide covers essential PDF processing operations using Python libraries and command-line tools. For       │
│  advanced features, JavaScript libraries, and detailed examples, see REFERENCE.md. If you need to fill out a    │
│  PDF form, read FORMS.md and follow its instructions.                                                           │
│                                                                                                                 │
│  Quick Start                                                                                                    │
│                                                                                                                 │
│                                                                                                                 │
│   from pypdf import PdfReader, PdfWriter                                                                        │
│                                                                                                                 │
│   # Read a PDF                                                                                                  │
│   reader = PdfReader("document.pdf")                                                                            │
│   print(f"Pages: {len(reader.pages)}")                                                                          │
│                                                                                                                 │
│   # Extract text                                                                                                │
│   text = ""                                                                                                     │
│   for page in reader.pages:                                                                                     │
│       text += page.extract_text()                                                                               │
│                                                                                                                 │
│                                                                                                                 │
│  Python Libraries                                                                                               │
│                                                                                                                 │
│  pypdf - Basic Operations                                                                                       │
│                                                                                                                 │
│  Merge PDFs                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
│   from pypdf import PdfWriter, PdfReader                                                                        │
│                                                                                                                 │
│   writer = PdfWriter()                                 


Frontmatter keys: ['name', 'description', 'license']
Body length: 7,513 chars (~1,878 tokens)


In [36]:
# What other files are bundled with the skill?
pdf_dir = Path("./skills/pdf")
from rich.table import Table
t = Table(title="Files bundled with the pdf skill")
t.add_column("Path", style="cyan")
t.add_column("Size", justify="right", style="yellow")
t.add_column("Loaded", style="green")

for f in sorted(pdf_dir.rglob("*")):
    if f.is_file():
        rel = f.relative_to(pdf_dir)
        size = f.stat().st_size
        if rel.name == "SKILL.md":
            loaded = "Tier 1+2 (always)"
        elif f.suffix == ".md":
            loaded = "Tier 3 (on reference)"
        elif f.suffix == ".py":
            loaded = "Executed only"
        else:
            loaded = "On demand"
        t.add_row(str(rel), f"{size:,} B", loaded)

console.print(t)

                        Files bundled with the pdf skill                        
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Path                                      ┃     Size ┃ Loaded                ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━┩
│ LICENSE.txt                               │  1,467 B │ On demand             │
│ SKILL.md                                  │  8,072 B │ Tier 1+2 (always)     │
│ forms.md                                  │ 11,854 B │ Tier 3 (on reference) │
│ reference.md                              │ 16,692 B │ Tier 3 (on reference) │
│ scripts/check_bounding_boxes.py           │  2,774 B │ Executed only         │
│ scripts/check_fillable_fields.py          │    268 B │ Executed only         │
│ scripts/convert_pdf_to_images.py          │  1,008 B │ Executed only         │
│ scripts/create_validation_image.py        │  1,258 B │ Executed only         │
│ scripts/extract_form_field_info.py        │  4,300 B │ Executed only         │
│ scripts/extract_form_structure.py         │  3,945 B │ Executed only         │
│ scripts/fill_fillable_fields.py           │  3,819 B │ Executed only         │
│ scripts/fill_pdf_form_with_annotations.py │  3,235 B │ Executed only         │
└───────────────────────────────────────────┴──────────┴───────────────────────┘

**Notice the loading tiers.** This is *progressive disclosure*:

| Tier | What loads | When | Cost |
|------|-----------|------|------|
| **1** | `name` + `description` only | Session start | ~50–100 tokens / skill |
| **2** | Full `SKILL.md` body | When the description matches | ~1K–5K tokens / skill |
| **3** | Bundled `reference.md`, etc. | When the body links to them | Per-file |
| **(Scripts)** | Python files | Executed by the agent, not loaded into context | 0 context tokens |

You can have **50 skills installed** and pay <5K tokens of overhead — they only "cost real tokens" the moment they fire.

---
## Section 2: From Ground-Up to Library

Same pedagogy as Sessions 2 and 3: **build it by hand, then use the library**. We'll write a tiny SKILL.md loader ourselves so the mechanics are obvious, then switch to `create_deep_agent(skills=[...])` in one line.

### Section 2a: Hand-rolled SKILL.md loader

In [37]:
def load_skills_manually(skills_dir: str) -> tuple[str, str]:
    """Mimic Tier 1 progressive disclosure: build a system-prompt section listing every installed skill by name+description."""
    descriptions = []
    bodies = {}
    for sd in sorted(Path(skills_dir).iterdir()):
        skill_md = sd / "SKILL.md"
        if not skill_md.exists():
            continue
        s = parse_skill_md(str(skill_md))
        name = s["frontmatter"].get("name", sd.name)
        desc = s["frontmatter"].get("description", "")
        descriptions.append(f"- **{name}**: {desc[:200]}{'...' if len(desc) > 200 else ''}")
        bodies[name] = s["body"]
    catalog = "## Available Skills\n\n" + "\n".join(descriptions)
    catalog += "\n\nWhen the user's request matches a skill description, call the load_skill_body tool to read the full SOP."
    return catalog, bodies

manual_catalog, manual_bodies = load_skills_manually("./skills")
show_prompt(manual_catalog, title="Hand-rolled Tier-1 catalog")

╭────────────────────────────────────────── Hand-rolled Tier-1 catalog ───────────────────────────────────────────╮
│                                                                                                                 │
│  ## Available Skills                                                                                            │
│                                                                                                                 │
│  - **daily-standup**: Generate a clean markdown standup post for Slack from yesterday's work, today's plan,     │
│  and blockers. Use this skill whenever the user wants to write a daily standup, daily sync, or team update —    │
│  even i...                                                                                                      │
│  - **pdf**: Use this skill whenever the user wants to do anything with PDF files. This includes reading or      │
│  extracting text/tables from PDFs, combining or merging multiple PDFs into one, splitting PDFs apart, rota...   │
│  - **release-notes**: Use this skill whenever the user asks for release notes, a changelog entry, a version     │
│  summary, or a 'what shipped' post from a list of raw commit messages, PR titles, or change descriptions.       │
│  Triggers ...                                                                                                   │
│  - **skill-creator**: Create new skills, modify and improve existing skills, and measure skill performance.     │
│  Use when users want to create a skill from scratch, edit, or optimize an existing skill, run evals to test a   │
│  skill...                                                                                                       │
│  - **slide-deck**: Use this skill whenever the user asks for a slide deck, presentation, slides, deck, talk     │
│  outline, or webinar content. Produces a clean MARKDOWN slide deck (NOT HTML) — one slide per H2 heading, with  │
│  s...                                                                                                           │
│  - **web-research**: Use this skill whenever the user asks to research a topic, gather information, find        │
│  sources, or produce a brief on something. Triggers on phrases like "research", "find out about",               │
│  "investigate", "sum...                                                                                         │
│  - **xlsx**: Use this skill any time a spreadsheet file is the primary input or output. This means any task     │
│  where the user wants to: open, read, edit, or fix an existing .xlsx, .xlsm, .csv, or .tsv file (e.g., add...   │
│                                                                                                                 │
│  When the user's request matches a skill description, call the load_skill_body tool to read the full SOP.       │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [38]:
# Tier-1 token cost: ~100 tokens per skill description
from rich.table import Table
t = Table(title="Token cost of Tier-1 catalog")
t.add_column("Source", style="cyan")
t.add_column("Chars", justify="right", style="yellow")
t.add_column("~Tokens", justify="right", style="green")

t.add_row("Tier-1 catalog (all skills, descriptions only)", f"{len(manual_catalog):,}", f"{len(manual_catalog)//4:,}")
for name, body in manual_bodies.items():
    t.add_row(f"   ↳ Tier-2 body for '{name}' (loaded only on match)", f"{len(body):,}", f"{len(body)//4:,}")
console.print(t)

print("\nWith progressive disclosure: we pay the small Tier-1 cost up front; large Tier-2 bodies only load on match.")

                           Token cost of Tier-1 catalog                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━━━┓
┃ Source                                                      ┃  Chars ┃ ~Tokens ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━━━┩
│ Tier-1 catalog (all skills, descriptions only)              │  1,678 │     419 │
│    ↳ Tier-2 body for 'daily-standup' (loaded only on match) │  2,003 │     500 │
│    ↳ Tier-2 body for 'pdf' (loaded only on match)           │  7,513 │   1,878 │
│    ↳ Tier-2 body for 'release-notes' (loaded only on match) │  2,230 │     557 │
│    ↳ Tier-2 body for 'skill-creator' (loaded only on match) │ 32,626 │   8,156 │
│    ↳ Tier-2 body for 'slide-deck' (loaded only on match)    │  2,459 │     614 │
│    ↳ Tier-2 body for 'web-research' (loaded only on match)  │    644 │     161 │
│    ↳ Tier-2 body for 'xlsx' (loaded only on match)          │ 10,424 │   2,606 │
└─────────────────────────────────────────────────────────────┴────────┴─────────┘


With progressive disclosure: we pay the small Tier-1 cost up front; large Tier-2 bodies only load on match.


### Section 2b: Library — `create_deep_agent(skills=[...])`

All of the above is built into `deepagents` v0.5.9+. One parameter, zero glue code.

In [39]:
from deepagents import create_deep_agent
# Already imported above: FilesystemBackend, ChatSambaNova, MODEL, BACKEND

agent_with_skills = create_deep_agent(
    model=MODEL,
    system_prompt="You are a capable assistant. Use the available skills when they match the user's request.",
    tools=[],
    backend=BACKEND,            # ← mount the project dir
    skills=["/skills"],          # ← POSIX path INSIDE the backend
)

print("✅ Agent created with native skills support.")
print(f"   Backend: FilesystemBackend(root_dir='.')")
print(f"   Skill sources: /skills (resolves to ./skills/ on disk)")
print(f"   Detected: {sorted(d for d in os.listdir(SKILLS_DIR) if os.path.isdir(os.path.join(SKILLS_DIR, d)))}")


✅ Agent created with native skills support.
   Backend: FilesystemBackend(root_dir='.')
   Skill sources: /skills (resolves to ./skills/ on disk)
   Detected: ['daily-standup', 'pdf', 'release-notes', 'skill-creator', 'slide-deck', 'web-research', 'xlsx']


In [40]:
# Confirm SkillsMiddleware is in the agent's middleware stack
from deepagents.middleware.skills import SkillsMiddleware

node_names = list(agent_with_skills.get_graph().nodes.keys())
print("Graph nodes:", node_names)
print("\nSkillsMiddleware is wired in by `skills=[...]` — descriptions injected at startup, bodies loaded on match, scripts execute via filesystem tools.")

Graph nodes: ['__start__', 'model', 'tools', 'TodoListMiddleware.after_model', 'SkillsMiddleware.before_agent', 'PatchToolCallsMiddleware.before_agent', '__end__']

SkillsMiddleware is wired in by `skills=[...]` — descriptions injected at startup, bodies loaded on match, scripts execute via filesystem tools.


---
## Section 3: Demo 1 — Skill Recognition & SOP Execution

Two parts to this demo:

**3a (Tier 1 in action):** the agent reports which of its installed skills it would use for different requests — without loading any skill bodies. This shows progressive disclosure: the agent ROUTES on descriptions, then decides what to load.

**3b (Tier 2 execution):** we author-include a `release-notes` skill — a pure-SOP business-logic skill (no scripts, no code execution required). The agent loads the body and follows the standard operating procedure to turn raw commit messages into a clean changelog. This is the canonical pattern for **style guides, SOPs, and business-logic skills** — most production skills look like this.

### Section 3a — Skill Recognition (no body loading)

In [41]:
router_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a skill router. For each user request, name the single best skill from your "
        "installed catalog and quote the one phrase from its description that matches. "
        "DO NOT load the skill body — just route."
    ),
    tools=[],
    backend=BACKEND,
    skills=["/skills"],
)

print("Routing 4 different requests across the catalog...\n")
result_router = router_agent.invoke({
    "messages": [{"role": "user", "content": (
        "For each of these requests, tell me which installed skill matches and the trigger phrase:\n"
        "  1. 'Build a 5-slide talk for our product team.'\n"
        "  2. 'Turn this list of merged PRs into release notes for v2.4.'\n"
        "  3. 'I need to research how MCP servers handle auth.'\n"
        "  4. 'Author a new skill that drafts incident postmortems.'\n"
    )}]
}, config={"recursion_limit": 6})

format_messages(result_router["messages"])

Routing 4 different requests across the catalog...



╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ For each of these requests, tell me which installed skill matches and the trigger phrase:                       │
│   1. 'Build a 5-slide talk for our product team.'                                                               │
│   2. 'Turn this list of merged PRs into release notes for v2.4.'                                                │
│   3. 'I need to research how MCP servers handle auth.'                                                          │
│   4. 'Author a new skill that drafts incident postmortems.'                                                     │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ | # | Request | Matching Skill | Trigger Phrase |                                                               │
│ |---|---------|----------------|----------------|                                                               │
│ | 1 | Build a 5-slide talk for our product team. | **slide-deck** | "slide deck, presentation, slides, deck,    │
│ talk outline, or webinar content" |                                                                             │
│ | 2 | Turn this list of merged PRs into release notes for v2.4. | **release-notes** | "release notes, a         │
│ changelog entry, a version summary, or a 'what shipped' post" |                                                 │
│ | 3 | I need to research how MCP servers handle auth. | **web-research** | "research a topic, gather            │
│ information, find sources, or produce a brief on something" |                                                   │
│ | 4 | Author a new skill that drafts incident postmortems. | **skill-creator** | "Create new skills, modify and │
│ improve existing skills, and measure skill performance" |                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**What we just saw:** the agent matched each request to the right installed skill *without* reading any SKILL.md body. That's **Tier 1** progressive disclosure — descriptions only, ~100 tokens per skill, the routing is cheap.


### Section 3b — SOP-Style Skill Execution (`release-notes`)

In [42]:
# First, peek at the release-notes SKILL.md so the audience sees what the agent will follow
pretty_skill("./skills/release-notes/SKILL.md")

╭─────────────────────────────── 📄 ./skills/release-notes/SKILL.md — frontmatter ────────────────────────────────╮
│ name: release-notes                                                                                             │
│ description: "Use this skill whenever the user asks for release notes, a changelog entry, a version summary, or │
│ a 'what shipped' post from a list of raw commit messages, PR titles, or change descriptions. Triggers on        │
│ phrases like 'release notes', 'changelog', 'what's new', 'version notes', 'changes for vX.Y'. The deliverable   │
│ is a single structured markdown document with Features / Fixes / Breaking sections. Do NOT trigger for code     │
│ reviews, design docs, or freeform writing tasks."                                                               │
│ allowed-tools: write_file, read_file, ls                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── 📄 ./skills/release-notes/SKILL.md — body ───────────────────────────────────╮
│                                                                                                                 │
│                                              Release-Notes Builder                                              │
│                                                                                                                 │
│  Output format                                                                                                  │
│                                                                                                                 │
│  A single markdown file at ./release_notes_<version>.md. The structure is fixed:                                │
│                                                                                                                 │
│                                                                                                                 │
│   # Release vX.Y.Z — <short headline>                                                                           │
│                                                                                                                 │
│   _Released <YYYY-MM-DD>_                                                                                       │
│                                                                                                                 │
│   ## ✨ Features                                                                                                │
│   - Concrete user-facing change (12 words max, active voice)                                                    │
│                                                                                                                 │
│   ## 🐛 Fixes                                                                                                   │
│   - What was broken, now fixed                                                                                  │
│                                                                                                                 │
│   ## ⚠️ Breaking changes                                                                                        │
│   - What changed, what users must do to adapt                                                                   │
│                                                                                                                 │
│   ## 🧰 Internal                                                                                                │
│   - Refactors, deps bumps, infra changes                                                                        │
│                                                                                                                 │
│   ## Contributors                                                                                               │
│   - @handle1, @handle2                                                                                          │
│                                                                                                                 │
│   ---                                                                                                           │
│                                                                                                                 │
│   _Full diff: <link or commit range>_                                                                           │
│                                                                                                                 │
│                                                                                                                 │
│  If a section has no entries, omit it entirely — don't write "None".                                            │
│                                                           

In [43]:
# Now: build an agent and have it execute the release-notes SOP on real commit messages.
# This is a pure-text skill — no scripts, no code execution. The agent reads the SOP and follows it.

release_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You produce release deliverables. Use the available skills.",
    tools=[],
    backend=BACKEND,
    skills=["/skills"],
)

SAMPLE_COMMITS = '''
feat(auth): add SAML SSO support for enterprise plans (#1234)
feat(api): introduce /v2/search endpoint with pagination (#1240)
fix(api): correct race condition in concurrent session refresh (#1241)
fix(ui): resolve overflow on long names in user picker (#1245)
fix(api): handle empty body in PATCH requests gracefully (#1247)
BREAKING: remove deprecated /v1/login endpoint (use /v2/auth) (#1250)
chore(deps): bump langchain-core to 1.4.0
refactor(internal): consolidate logging into structured-log middleware
feat(observability): add request-id header to every response (#1252)
fix(billing): correct proration calc when downgrading mid-cycle (#1253)
'''

print("Running release-notes skill on sample commit list...\n")
result_release = release_agent.invoke({
    "messages": [{"role": "user", "content": (
        f"Produce release notes for v2.4.0 (release date 2026-05-13) from these commits. "
        f"Use the release-notes skill — follow its SOP exactly. Save the file at /outputs/release_notes_v2.4.0.md (create the outputs/ dir if missing).\n\n"
        f"Commits:\n{SAMPLE_COMMITS}"
    )}]
}, config={"recursion_limit": 15})

Running release-notes skill on sample commit list...



In [44]:
# Render the produced release notes inline
from pathlib import Path
import time
since = time.time() - 300
candidates = sorted([p for p in Path('outputs').glob('release_notes*.md')] if Path('outputs').is_dir() else [])
if not candidates:
    candidates = sorted(p for p in Path('.').glob('release_notes*.md'))
print(f"Release-notes files on disk: {[str(p) for p in candidates]}\n")
for p in candidates[-1:]:
    text = p.read_text()
    pretty_md(text, title=f"📝 {p.name} ({len(text):,} chars)")

Release-notes files on disk: ['release_notes_v2.4.0.md']



╭──────────────────────────────────── 📝 release_notes_v2.4.0.md (790 chars) ─────────────────────────────────────╮
│                                                                                                                 │
│                               Release v2.4.0 — SAML SSO, New Search API, and More                               │
│                                                                                                                 │
│  Released 2026-05-13                                                                                            │
│                                                                                                                 │
│  ✨ Features                                                                                                    │
│                                                                                                                 │
│   • Added SAML SSO support for enterprise plans (#1234)                                                         │
│   • Introduced /v2/search endpoint with pagination (#1240)                                                      │
│   • Added request-id header to every API response for tracing (#1252)                                           │
│                                                                                                                 │
│  🐛 Fixes                                                                                                       │
│                                                                                                                 │
│   • Fixed race condition in concurrent session refresh (#1241)                                                  │
│   • Resolved overflow on long names in user picker (#1245)                                                      │
│   • Fixed empty body handling in PATCH requests (#1247)                                                         │
│   • Corrected proration calculation when downgrading mid-cycle (#1253)                                          │
│                                                                                                                 │
│  ⚠️ Breaking changes                                                                                            │
│                                                                                                                 │
│   • Removed deprecated /v1/login endpoint — migrate to /v2/auth (#1250)                                         │
│                                                                                                                 │
│  🧰 Internal                                                                                                    │
│                                                                                                                 │
│   • Bumped langchain-core to 1.4.0                                                                              │
│   • Consolidated logging into structured-log middleware                                                         │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Full diff: https://github.com/org/repo/compare/v2.3.0...v2.4.0                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**What we just saw:** the agent loaded `release-notes/SKILL.md` (the SOP), recognized that we passed in commit messages, classified each one into Features / Fixes / Breaking / Internal, rewrote them in plain user-facing language, and saved a properly-formatted changelog file. **No code execution.** Just an agent following a markdown SOP.

This is the pattern for the majority of production skills: **style guides, SOPs, business-logic procedures**. The SKILL.md is the contract; the agent is the executor.


---
## Section 4: Demo 2 — Authoring a Custom Skill

We don't need to wait for someone else to publish a skill for our workflow. Let's **author one inline** — a `slide-deck` skill that builds a markdown deck from a topic and audience.

Total surface area: one `SKILL.md`. No code.

**Note:** we already shipped `./skills/slide-deck/SKILL.md` with this notebook. Inspect it inline.

In [45]:
# Show the slide-deck SKILL.md — this is the entire skill
slide_skill = parse_skill_md("./skills/slide-deck/SKILL.md")
console.print(Panel.fit(
    "\n".join(f"[bold cyan]{k}[/]: {v}" for k, v in slide_skill["frontmatter"].items()),
    title="📊 slide-deck/SKILL.md frontmatter",
    border_style="magenta",
))
print(f"\nSKILL.md body (the procedure):\n{'-' * 60}\n{slide_skill['body'][:1200]}...")

╭────────────────────────────────────── 📊 slide-deck/SKILL.md frontmatter ───────────────────────────────────────╮
│ name: slide-deck                                                                                                │
│ description: Use this skill whenever the user asks for a slide deck, presentation, slides, deck, talk outline,  │
│ or webinar content. Produces a clean MARKDOWN slide deck (NOT HTML) — one slide per H2 heading, with speaker    │
│ notes inline. Use when the deliverable is a multi-slide presentation (3+ slides), the user mentions a target    │
│ audience or duration, or asks for a structured talk. Do NOT trigger for single-page summaries, prose documents, │
│ or one-off paragraphs. Output is markdown only — never HTML, never CSS, never JavaScript.                       │
│ allowed-tools: write_file, edit_file, read_file, ls                                                             │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯


SKILL.md body (the procedure):
------------------------------------------------------------

# Slide-Deck Builder

## Output format — read this first

**The output is ONE markdown file (`.md`).** Never HTML, never a CSS stylesheet, never a slide-rendering framework. Each slide is a markdown H2 heading. The total file should be **under 2,000 words** — short slides are better slides.

## Overview

Build a markdown slide deck (`.md` file in the virtual filesystem) from a research brief, topic, or outline. One H2 heading per slide. Speaker notes go in a blockquote under each slide.

## Standard Operating Procedure

1. **Clarify scope** if missing: target audience, duration (minutes), tone (technical / executive / educational), preferred slide count.
2. **Draft an outline** — list every slide title before writing content. Stop and confirm with the user if scope is >12 slides or the topic is ambiguous.
3. **Write each slide** using the template below. Keep slide bodies to ≤5 bullet points o

In [46]:
# Rebuild the agent — it auto-discovers the slide-deck skill
slide_agent = create_deep_agent(
    model=MODEL,
    system_prompt="You are a helpful assistant. Use the available skills when they match the user's request.",
    tools=[],
    backend=BACKEND,
    skills=["/skills"],
)

print("Running agent on slide-deck authoring task...\n")
result_slides = slide_agent.invoke({
    "messages": [
        {"role": "user", "content": (
            "Build a 5-slide MARKDOWN deck on transformer architectures for an engineering audience. "
            "Target 10 minutes total. Save it to /outputs/transformer_slides.md. Output a single .md file using H2 headings per slide — no HTML."
        )}
    ]
})


Running agent on slide-deck authoring task...



In [47]:
# With FilesystemBackend, write_file lands on disk in this directory.
# Find the slide-deck file the agent produced.
from pathlib import Path
import time
since = time.time() - 600  # within last 10 min
candidates = sorted(Path('outputs').glob('*.md')) if Path('outputs').is_dir() else []
candidates = [p for p in candidates if p.stat().st_mtime > since and ('slide' in p.name.lower() or 'transformer' in p.name.lower() or 'deck' in p.name.lower())]

print(f"Slide-deck files produced: {[str(p) for p in candidates]}\n")
for p in candidates:
    text = p.read_text()
    pretty_md(text, title=f"📊 {p.name} ({len(text):,} chars)")


Slide-deck files produced: ['transformer_architectures_slides.md', 'transformer_architectures_engineering_slides.md', 'transformer_arch_deck_final.md', 'transformers_engineering_5slide.md', 'transformer_eng_deck.md', 'transformer_slides_5deck.md', 'transformer_arch_eng_2025.md', 'transformer_deck.md']



╭───────────────────────────── 📊 transformer_architectures_slides.md (3,356 chars) ──────────────────────────────╮
│                                                                                                                 │
│  Slide 1 — Transformers Beat RNNs by Learning What to Attend To                                                 │
│                                                                                                                 │
│   • Sequence-to-sequence tasks dominated by RNNs for a decade                                                   │
│   • RNNs process tokens sequentially — no parallelization, vanishing gradients                                  │
│   • Transformers replace recurrence with direct token-to-token attention                                        │
│   • Result: parallelizable training, long-range dependencies, state-of-the-art on every NLP benchmark           │
│                                                                                                                 │
│  ▌ Speaker notes: Open with the core problem Transformers solved. RNNs were the default for sequence tasks      │
│  ▌ but had fundamental limits — they couldn't parallelize and struggled with long contexts. Engineers need      │
│  ▌ to understand why this architectural shift happened before diving into mechanics.                            │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: The Core Innovation                                                                  │
│                                                                                                                 │
│   • Each token attends to all other tokens in the sequence simultaneously                                       │
│   • Attention score = similarity between query and key vectors, scaled by √d_k                                  │
│   • Weighted sum of values produces a context-aware representation per token                                    │
│   • No recurrence, no convolution — pure matrix multiplication                                                  │
│                                                                                                                 │
│  ▌ Speaker notes: This is the heart of the transformer. Walk through the Q, K, V intuition carefully — it's     │
│  ▌ the single concept everything else builds on. Emphasize that this is O(n²) in sequence length but fully      │
│  ▌ parallelizable, unlike RNN's O(n) sequential chain.                                                          │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 3 — Multi-Head Attention and Positional Encoding                                                         │
│                                                                                                                 │
│   • Multiple attention heads run in parallel, each learning different relationship types                        │
│   • Heads concatenate and project — model captures syntax, semantics, and coreference jointly                   │
│   • Positional encoding injects sequence order via fixed sinusoidal or learned embeddings                       │
│   • Without positional info, the architecture is permutation-invariant — order would be lost                    │
│                                                        

╭─────────────────────── 📊 transformer_architectures_engineering_slides.md (4,637 chars) ────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

╭──────────────────────────────── 📊 transformer_arch_deck_final.md (4,637 chars) ────────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

╭────────────────────────────── 📊 transformers_engineering_5slide.md (4,637 chars) ──────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

╭─────────────────────────────────── 📊 transformer_eng_deck.md (4,637 chars) ────────────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

╭───────────────────────────────── 📊 transformer_slides_5deck.md (4,637 chars) ──────────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

╭───────────────────────────────── 📊 transformer_arch_eng_2025.md (4,637 chars) ─────────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

╭───────────────────────────────────── 📊 transformer_deck.md (4,637 chars) ──────────────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu

In [48]:
format_messages(result_slides["messages"])

╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Build a 5-slide MARKDOWN deck on transformer architectures for an engineering audience. Target 10 minutes       │
│ total. Output a single .md file using H2 headings per slide — no HTML.                                          │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: read_file                                                                                         │
│    Args: {                                                                                                      │
│   "file_path": "/skills/slide-deck/SKILL.md",                                                                   │
│   "limit": 200                                                                                                  │
│ }                                                                                                               │
│    ID: call_960f8cac15f24cc3a5                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│      1  ---                                                                                                     │
│      2  name: slide-deck                                                                                        │
│      3  description: "Use this skill whenever the user asks for a slide deck, presentation, slides, deck, talk  │
│ outline, or webinar content. Produces a clean MARKDOWN slide deck (NOT HTML) — one slide per H2 heading, with   │
│ speaker notes inline. Use when the deliverable is a multi-slide presentation (3+ slides), the user mentions a   │
│ target audience or duration, or asks for a structured talk. Do NOT trigger for single-page summaries, prose     │
│ documents, or one-off paragraphs. Output is markdown only — never HTML, never CSS, never JavaScript."           │
│      4  allowed-tools: write_file, edit_file, read_file, ls                                                     │
│      5  ---                                                                                                     │
│      6                                                                                                          │
│      7  # Slide-Deck Builder                                                                                    │
│      8                                                                                                          │
│      9  ## Output format — read this first                                                                      │
│     10                                                                                                          │
│     11  **The output is ONE markdown file (`.md`).** Never HTML, never a CSS stylesheet, never a                │
│ slide-rendering framework. Each slide is a markdown H2 heading. The total file should be **under 2,000 words**  │
│ — short slides are better slides.                                                                               │
│     12                                                                                                          │
│     13  ## Overview                                                                                             │
│     14                                                                                                          │
│     15  Build a markdown slide deck (`.md` file in the virtual filesystem) from a research brief, topic, or     │
│ outline. One H2 heading per slide. Speaker notes go in a blockquote under each slide.                           │
│     16                                                                                                          │
│     17  ## Standard Operating Procedure                                                                         │
│     18                                                                                                          │
│     19  1. **Clarify scope** if missing: target audience, duration (minutes), tone (technical / executive /     │
│ educational), preferred slide count.                                                                            │
│     20  2. **Draft an outline** — list every slide title before writing content. Stop and confirm with the user │
│ if scope is >12 slides or the topic is ambiguous.                                                               │
│     21  3. **Write each slide** using the template below. Keep slide bodies to ≤5 bullet points or 1 diagram    │
│ description. The audience reads slides in <30 seconds.                                                          │
│     22  4. **Add speaker notes** under each slide as a `> Speaker notes:` blockquote — 2–4 sentences max, the   │
│ *why* not the *what*.                                                                                           │
│     23  5. **Save** the deck to a single file: `./<topi

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: write_file                                                                                        │
│    Args: {                                                                                                      │
│   "content": "# Transformer Architectures — Engineering Deep Dive\n\n---\n\n## Slide 1 — The Attention          │
│ Revolution\n\n- RNNs struggled with long-range dependencies and sequential bottlenecks\n- Transformers replace  │
│ recurrence with parallel, learned attention\n- \"Attention is All You Need\" (Vaswani et al., 2017) launched a  │
│ paradigm shift\n- Key insight: every token attends to every other token in constant time\n\n> Speaker notes:    │
│ Engineers had been fighting RNN limitations for years — vanishing gradients, sequential processing that blocked │
│ GPU parallelism. Transformers solved both at once. This slide establishes the *why*, not just the *what*.       │
│ Engineers in the room have likely used a transformer-based model; this frames the underlying                    │
│ mechanism.\n\n---\n\n## Slide 2 — Self-Attention: Q, K, V Mechanics\n\n- Each input token is projected into     │
│ **Query (Q)**, **Key (K)**, and **Value (V)** vectors\n- Attention score = softmax(QKᵀ / √dₖ) · V\n- √dₖ        │
│ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions\n- Result: a weighted      │
│ context vector per token, learned from data\n\n```\nAttention(Q, K, V) = softmax(QKᵀ / √dₖ) · V\n```\n\n>       │
│ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks \"what do I need?\", K     │
│ answers \"what do I offer?\", and V is the actual content. The softmax normalizes across all positions,         │
│ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —       │
│ without it, dot products grow large and softmax saturates to one-hot distributions.\n\n---\n\n## Slide 3 —      │
│ Multi-Head Attention & Positional Encoding\n\n- Multiple attention heads run in parallel, each with independent │
│ Q/K/V projections\n- Heads learn different relationship types (syntax, semantics, proximity) simultaneously\n-  │
│ Outputs are concatenated and projected: **MultiHead(Q,K,V) = Concat(head₁,…,headₕ)Wᴼ**\n- Positional encoding   │
│ adds sinusoidal or learned position signals — transformers have no inherent order\n\n> Speaker notes:           │
│ Multi-head attention is where transformers earn their parallelism. Each head independently computes attention,  │
│ then the results are fused. This is embarrassingly parallel — a GPU engineer's dream. Positional encoding is a  │
│ subtle but critical hack: without it, the architecture is permutation-invariant and can't distinguish \"the cat │
│ sat\" from \"sat the cat.\" Engineers should note that rotary embeddings (RoPE) and ALiBi have largely replaced │
│ absolute sinusoidal encodings in modern models.\n\n---\n\n## Slide 4 — Stacking Layers: Encoder, Decoder, and   │
│ FFN\n\n- A transformer block = Multi-Head Attention + Feed-Forward Network (FFN) + Residual connections +       │
│ LayerNorm\n- **Encoder**: bidirectional context — every token sees all others (BERT, ViT)\n- **Decoder**:       │
│ causal masking — each token sees itself and prior tokens (GPT, Llama)\n- Stacking N layers compounds            │
│ representational power — GPT-3 uses 96 layers, 175B parameters\n\n```\nTransformerBlock = LayerNorm(x +         │
│ MultiHeadAttention(x)) + LayerNorm(x + FFN(x))\n```\n\n> Speaker notes: The FFN is often overlooked but         │
│ accounts for ~2/3 of transformer parameters. It's two li

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Cannot write to /transformer_architectures_slides.md because it already exists. Read and then make an edit, or  │
│ write to a new path.                                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

...

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: write_file                                                                                        │
│    Args: {                                                                                                      │
│   "content": "# Transformer Architectures — Engineering Deep Dive\n\n---\n\n## Slide 1 — The Attention          │
│ Revolution\n\n- RNNs struggled with long-range dependencies and sequential bottlenecks\n- Transformers replace  │
│ recurrence with parallel, learned attention\n- \"Attention is All You Need\" (Vaswani et al., 2017) launched a  │
│ paradigm shift\n- Key insight: every token attends to every other token in constant time\n\n> Speaker notes:    │
│ Engineers had been fighting RNN limitations for years — vanishing gradients, sequential processing that blocked │
│ GPU parallelism. Transformers solved both at once. This slide establishes the *why*, not just the *what*.       │
│ Engineers in the room have likely used a transformer-based model; this frames the underlying                    │
│ mechanism.\n\n---\n\n## Slide 2 — Self-Attention: Q, K, V Mechanics\n\n- Each input token is projected into     │
│ **Query (Q)**, **Key (K)**, and **Value (V)** vectors\n- Attention score = softmax(QKᵀ / √dₖ) · V\n- √dₖ        │
│ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions\n- Result: a weighted      │
│ context vector per token, learned from data\n\n```\nAttention(Q, K, V) = softmax(QKᵀ / √dₖ) · V\n```\n\n>       │
│ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks \"what do I need?\", K     │
│ answers \"what do I offer?\", and V is the actual content. The softmax normalizes across all positions,         │
│ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —       │
│ without it, dot products grow large and softmax saturates to one-hot distributions.\n\n---\n\n## Slide 3 —      │
│ Multi-Head Attention & Positional Encoding\n\n- Multiple attention heads run in parallel, each with independent │
│ Q/K/V projections\n- Heads learn different relationship types (syntax, semantics, proximity) simultaneously\n-  │
│ Outputs are concatenated and projected: **MultiHead(Q,K,V) = Concat(head₁,…,headₕ)Wᴼ**\n- Positional encoding   │
│ adds sinusoidal or learned position signals — transformers have no inherent order\n\n> Speaker notes:           │
│ Multi-head attention is where transformers earn their parallelism. Each head independently computes attention,  │
│ then the results are fused. This is embarrassingly parallel — a GPU engineer's dream. Positional encoding is a  │
│ subtle but critical hack: without it, the architecture is permutation-invariant and can't distinguish \"the cat │
│ sat\" from \"sat the cat.\" Engineers should note that rotary embeddings (RoPE) and ALiBi have largely replaced │
│ absolute sinusoidal encodings in modern models.\n\n---\n\n## Slide 4 — Stacking Layers: Encoder, Decoder, and   │
│ FFN\n\n- A transformer block = Multi-Head Attention + Feed-Forward Network (FFN) + Residual connections +       │
│ LayerNorm\n- **Encoder**: bidirectional context — every token sees all others (BERT, ViT)\n- **Decoder**:       │
│ causal masking — each token sees itself and prior tokens (GPT, Llama)\n- Stacking N layers compounds            │
│ representational power — GPT-3 uses 96 layers, 175B parameters\n\n```\nTransformerBlock = LayerNorm(x +         │
│ MultiHeadAttention(x)) + LayerNorm(x + FFN(x))\n```\n\n> Speaker notes: The FFN is often overlooked but         │
│ accounts for ~2/3 of transformer parameters. It's two li

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Cannot write to /xformer_eng_slides.md because it already exists. Read and then make an edit, or write to a new │
│ path.                                                                                                           │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: write_file                                                                                        │
│    Args: {                                                                                                      │
│   "content": "# Transformer Architectures — Engineering Deep Dive\n\n---\n\n## Slide 1 — The Attention          │
│ Revolution\n\n- RNNs struggled with long-range dependencies and sequential bottlenecks\n- Transformers replace  │
│ recurrence with parallel, learned attention\n- \"Attention is All You Need\" (Vaswani et al., 2017) launched a  │
│ paradigm shift\n- Key insight: every token attends to every other token in constant time\n\n> Speaker notes:    │
│ Engineers had been fighting RNN limitations for years — vanishing gradients, sequential processing that blocked │
│ GPU parallelism. Transformers solved both at once. This slide establishes the *why*, not just the *what*.       │
│ Engineers in the room have likely used a transformer-based model; this frames the underlying                    │
│ mechanism.\n\n---\n\n## Slide 2 — Self-Attention: Q, K, V Mechanics\n\n- Each input token is projected into     │
│ **Query (Q)**, **Key (K)**, and **Value (V)** vectors\n- Attention score = softmax(QKᵀ / √dₖ) · V\n- √dₖ        │
│ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions\n- Result: a weighted      │
│ context vector per token, learned from data\n\n```\nAttention(Q, K, V) = softmax(QKᵀ / √dₖ) · V\n```\n\n>       │
│ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks \"what do I need?\", K     │
│ answers \"what do I offer?\", and V is the actual content. The softmax normalizes across all positions,         │
│ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —       │
│ without it, dot products grow large and softmax saturates to one-hot distributions.\n\n---\n\n## Slide 3 —      │
│ Multi-Head Attention & Positional Encoding\n\n- Multiple attention heads run in parallel, each with independent │
│ Q/K/V projections\n- Heads learn different relationship types (syntax, semantics, proximity) simultaneously\n-  │
│ Outputs are concatenated and projected: **MultiHead(Q,K,V) = Concat(head₁,…,headₕ)Wᴼ**\n- Positional encoding   │
│ adds sinusoidal or learned position signals — transformers have no inherent order\n\n> Speaker notes:           │
│ Multi-head attention is where transformers earn their parallelism. Each head independently computes attention,  │
│ then the results are fused. This is embarrassingly parallel — a GPU engineer's dream. Positional encoding is a  │
│ subtle but critical hack: without it, the architecture is permutation-invariant and can't distinguish \"the cat │
│ sat\" from \"sat the cat.\" Engineers should note that rotary embeddings (RoPE) and ALiBi have largely replaced │
│ absolute sinusoidal encodings in modern models.\n\n---\n\n## Slide 4 — Stacking Layers: Encoder, Decoder, and   │
│ FFN\n\n- A transformer block = Multi-Head Attention + Feed-Forward Network (FFN) + Residual connections +       │
│ LayerNorm\n- **Encoder**: bidirectional context — every token sees all others (BERT, ViT)\n- **Decoder**:       │
│ causal masking — each token sees itself and prior tokens (GPT, Llama)\n- Stacking N layers compounds            │
│ representational power — GPT-3 uses 96 layers, 175B parameters\n\n```\nTransformerBlock = LayerNorm(x +         │
│ MultiHeadAttention(x)) + LayerNorm(x + FFN(x))\n```\n\n> Speaker notes: The FFN is often overlooked but         │
│ accounts for ~2/3 of transformer parameters. It's two li

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ Updated file /transformer_slides_5deck.md                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ Done. The deck is saved at `/transformer_slides_5deck.md`.                                                      │
│                                                                                                                 │
│ **5 slides · ~10 minutes (2 min/slide)**                                                                        │
│                                                                                                                 │
│ | # | Title | Core content |                                                                                    │
│ |---|-------|-------------|                                                                                     │
│ | 1 | The Attention Revolution | Why transformers replaced RNNs — parallelization, long-range deps |            │
│ | 2 | Self-Attention: Q, K, V Mechanics | The attention equation, √dₖ scaling, context vectors |                │
│ | 3 | Multi-Head Attention & Positional Encoding | Parallel heads, concat/project, RoPE/ALiBi as modern         │
│ alternatives |                                                                                                  │
│ | 4 | Stacking Layers: Encoder, Decoder, and FFN | Block anatomy, encoder vs decoder, FFN as 2/3 of params |    │
│ | 5 | Engineering Implications | Scaling laws, KV-cache, FlashAttention, production takeaways |                 │
│                                                                                                                 │
│ Each slide has ≤4 bullets and a `> Speaker notes:` blockquote with the *why* and transition logic.              │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**That's it.** ~50 lines of markdown in `SKILL.md` and the agent follows the procedure end-to-end. No Python. No tool definitions. The skill body is the contract.

---
## Section 5: Demo 3 — Composition with Subagents

Recall from Session 3 the subagent pattern: a supervisor delegates heavy work to focused subagents with isolated contexts. Skills compose naturally on top of that — **each subagent can have its own scoped skill set**.

Pattern for this demo:
- **Supervisor** has only the `web-research` skill (we'll author it inline) and Tavily search
- **`report_writer` subagent** has the `slide-deck` and `pdf` skills
- Supervisor researches, then delegates report production to `report_writer`

In [49]:
# Author a web-research skill inline. Same pattern — one markdown file.
WEB_RESEARCH_SKILL = """---
name: web-research
description: Use this skill whenever the user asks to research a topic, gather information, find sources, or produce a brief on something. Triggers on phrases like "research", "find out about", "investigate", "summarize what's known about". The deliverable is a well-sourced markdown brief saved to a file. Do NOT trigger for simple factual questions answerable from the model's own knowledge.
allowed-tools: tavily_search, write_file, read_file, ls
---

# Web Research SOP

## Standard Operating Procedure

1. **Identify 3-5 sub-questions** that fully cover the topic. Write them as a todo list.
2. **Search each sub-question** using `tavily_search`. Take notes — do NOT paste raw search dumps.
3. **Save findings** to a markdown file `./research/<topic_slug>.md` with sections per sub-question and a Sources list at the bottom.
4. **Print a 2-sentence headline summary** to the user — the brief is in the file, not the chat.

## Quality bar

- Each claim has at least one cited source URL
- No more than 800 words total — this is a brief, not a thesis
- Open with the bottom line; details follow
"""

Path("./skills/web-research").mkdir(exist_ok=True)
Path("./skills/web-research/SKILL.md").write_text(WEB_RESEARCH_SKILL)
print("✅ Authored ./skills/web-research/SKILL.md inline")
print(f"   Size: {len(WEB_RESEARCH_SKILL):,} chars (~{len(WEB_RESEARCH_SKILL)//4:,} tokens)")

✅ Authored ./skills/web-research/SKILL.md inline
   Size: 1,121 chars (~280 tokens)


In [50]:
# Now compose: supervisor with web-research skill + a report_writer subagent with slide-deck & pdf skills
from langchain_tavily import TavilySearch

tavily_tool = TavilySearch(max_results=3)

report_writer_subagent = {
    "name": "report_writer",
    "description": "Produces final deliverables (slide decks, PDF summaries) from a research brief. Use this when research is complete and you need polished output.",
    "system_prompt": (
        "You are a report writer. You receive a research brief in a file. "
        "Use the slide-deck skill and the pdf skill to produce the requested deliverables."
    ),
    "tools": [],
    "skills": ["/skills/slide-deck", "/skills/pdf"],  # ← scoped skills per subagent
}

composed_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You are a research supervisor. Use the web-research skill to gather information, "
        "then delegate deliverable production to the report_writer subagent via the task tool. "
        "You don't write reports yourself — you research, then hand off."
    ),
    tools=[tavily_tool],
    backend=BACKEND,
    skills=["/skills/web-research"],  # supervisor only sees web-research
    subagents=[report_writer_subagent],
)

print("✅ Composed agent ready:")
print("   Supervisor skills: web-research")
print("   report_writer skills: slide-deck, pdf")


✅ Composed agent ready:
   Supervisor skills: web-research
   report_writer skills: slide-deck, pdf


In [51]:
# §5 invocation — tightened for faster runs (the live webinar can use a longer prompt)
print("Running composed agent on a deep-research task...\n")
result_composed = composed_agent.invoke({
    "messages": [
        {"role": "user", "content": (
            "Research Anthropic's Agent Skills spec (one Tavily query is enough). "
            "Save findings to /outputs/research/agent-skills.md. Then have the report_writer "
            "subagent produce a single deliverable: a 3-slide markdown deck at "
            "/outputs/deliverables/agent-skills-deck.md. Keep total output under 1,500 words."
        )}
    ]
}, config={"recursion_limit": 25})
print("✅ Composition complete")


Running composed agent on a deep-research task...

✅ Composition complete


In [52]:
# §5 outputs land on disk under ./deliverables/ and ./research/ (per the SOPs)
from pathlib import Path
import time
since = time.time() - 1800  # within last 30 min
disk_files = [p for p in Path('outputs').rglob('*.md')] if Path('outputs').is_dir() else []

from rich.table import Table
t = Table(title="Composition outcome (files written to disk)")
t.add_column("Path", style="cyan")
t.add_column("Size", style="green", justify="right")
for p in sorted(disk_files):
    t.add_row(str(p), f"{p.stat().st_size:,} B")
t.add_row("-- supervisor msgs --", str(len(result_composed['messages'])))
_console.print(t)

for p in sorted(disk_files):
    text = p.read_text()
    pretty_md(text[:2000] + ("\n\n…[truncated]" if len(text) > 2000 else ""),
              title=f"📄 {p.name} ({len(text):,} chars)")

print("\nCallback to Session 3: the supervisor never saw raw search results OR the pdf/slide-deck skill bodies. Both kinds of isolation working at once.")


  Composition outcome (files written to disk)  
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━┓
┃ Path                              ┃    Size ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━┩
│ deliverables/agent-skills-deck.md │ 3,960 B │
│ transformer_slides_5deck.md       │ 4,709 B │
│ -- supervisor msgs --             │      12 │
└───────────────────────────────────┴─────────┘

╭───────────────────────────────────── 📄 agent-skills-deck.md (3,944 chars) ─────────────────────────────────────╮
│                                                                                                                 │
│                              Agent Skills: Equipping AI Agents for the Real World                               │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1: What Are Agent Skills?                                                                                │
│                                                                                                                 │
│  Agent Skills is Anthropic's open standard for defining reusable, specialized capabilities that extend          │
│  Claude's functionality.                                                                                        │
│                                                                                                                 │
│  The Problem It Solves                                                                                          │
│                                                                                                                 │
│   • General-purpose AI agents often lack domain-specific expertise                                              │
│   • Specialized knowledge is difficult to share and reuse across teams                                          │
│   • No standard format existed for packaging agent capabilities                                                 │
│                                                                                                                 │
│  The Solution                                                                                                   │
│                                                                                                                 │
│  Agent Skills provides a standardized way to:                                                                   │
│                                                                                                                 │
│   • Package specialized knowledge and workflows                                                                 │
│   • Enable dynamic discovery of relevant capabilities                                                           │
│   • Share expertise across organizations and use cases                                                          │
│                                                                                                                 │
│  Industry Context                                                                                               │
│                                                                                                                 │
│  Similar to how MCP (Model Context Protocol) became the de facto standard for AI tool use, Agent Skills aims    │
│  to standardize how AI agents acquire and apply specialized knowledge.                                          │
│                                                                                                                 │
│  ▌ "Agent Skills is designed to make it easy to equip agents with the right capabilities at the right time."    │
│  ▌ — Anthropic Engineering                                                                                      │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                        

╭───────────────────────────────── 📄 transformer_slides_5deck.md (4,637 chars) ──────────────────────────────────╮
│                                                                                                                 │
│                                Transformer Architectures — Engineering Deep Dive                                │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 1 — The Attention Revolution                                                                             │
│                                                                                                                 │
│   • RNNs struggled with long-range dependencies and sequential bottlenecks                                      │
│   • Transformers replace recurrence with parallel, learned attention                                            │
│   • "Attention is All You Need" (Vaswani et al., 2017) launched a paradigm shift                                │
│   • Key insight: every token attends to every other token in constant time                                      │
│                                                                                                                 │
│  ▌ Speaker notes: Engineers had been fighting RNN limitations for years — vanishing gradients, sequential       │
│  ▌ processing that blocked GPU parallelism. Transformers solved both at once. This slide establishes the        │
│  ▌ why, not just the what. Engineers in the room have likely used a transformer-based model; this frames the    │
│  ▌ underlying mechanism.                                                                                        │
│                                                                                                                 │
│  -------------------------------------------------------------------------------------------------------------  │
│                                                                                                                 │
│  Slide 2 — Self-Attention: Q, K, V Mechanics                                                                    │
│                                                                                                                 │
│   • Each input token is projected into Query (Q), Key (K), and Value (V) vectors                                │
│   • Attention score = softmax(QKᵀ / √dₖ) · V                                                                    │
│   • √dₖ normalizes dot-product magnitude, preventing vanishing gradients in large dimensions                    │
│   • Result: a weighted context vector per token, learned from data                                              │
│                                                                                                                 │
│                                                                                                                 │
│   Attention(Q, K, V) = softmax(QKᵀ / √dₖ) · V                                                                   │
│                                                                                                                 │
│                                                                                                                 │
│  ▌ Speaker notes: This equation is the core of the transformer. Walk through it: Q asks "what do I need?", K    │
│  ▌ answers "what do I offer?", and V is the actual content. The softmax normalizes across all positions,        │
│  ▌ producing a probability distribution that weights relevant context. The √dₖ scaling is an empirical fix —    │
│  ▌ without it, dot products grow large and softmax satu


Callback to Session 3: the supervisor never saw raw search results OR the pdf/slide-deck skill bodies. Both kinds of isolation working at once.


In [ ]:
format_messages(result_composed["messages"])

---
## Section 6: Demo 4 — A Skill That Writes Skills

The natural next step: stop hand-writing `SKILL.md` files yourself. Anthropic ships a **skill-creator** skill — a skill whose job is to author new skills with correct frontmatter, body structure, and quality checks.

Below we give the agent both `skill-creator` and the existing skill library, and ask it to **author a brand-new `daily-standup` skill** that generates standup notes from a list of yesterday/today/blockers.

In [53]:
# Show the skill-creator's own frontmatter (so the audience sees what it triggers on)
pretty_skill("./skills/skill-creator/SKILL.md")


╭─────────────────────────────── 📄 ./skills/skill-creator/SKILL.md — frontmatter ────────────────────────────────╮
│ name: skill-creator                                                                                             │
│ description: Create new skills, modify and improve existing skills, and measure skill performance. Use when     │
│ users want to create a skill from scratch, edit, or optimize an existing skill, run evals to test a skill,      │
│ benchmark skill performance with variance analysis, or optimize a skill's description for better triggering     │
│ accuracy.                                                                                                       │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─────────────────────────────────── 📄 ./skills/skill-creator/SKILL.md — body ───────────────────────────────────╮
│                                                                                                                 │
│                                                  Skill Creator                                                  │
│                                                                                                                 │
│  A skill for creating new skills and iteratively improving them.                                                │
│                                                                                                                 │
│  At a high level, the process of creating a skill goes like this:                                               │
│                                                                                                                 │
│   • Decide what you want the skill to do and roughly how it should do it                                        │
│   • Write a draft of the skill                                                                                  │
│   • Create a few test prompts and run claude-with-access-to-the-skill on them                                   │
│   • Help the user evaluate the results both qualitatively and quantitatively                                    │
│      • While the runs happen in the background, draft some quantitative evals if there aren't any (if there     │
│        are some, you can either use as is or modify if you feel something needs to change about them). Then     │
│        explain them to the user (or if they already existed, explain the ones that already exist)               │
│      • Use the eval-viewer/generate_review.py script to show the user the results for them to look at, and      │
│        also let them look at the quantitative metrics                                                           │
│   • Rewrite the skill based on feedback from the user's evaluation of the results (and also if there are any    │
│     glaring flaws that become apparent from the quantitative benchmarks)                                        │
│   • Repeat until you're satisfied                                                                               │
│   • Expand the test set and try again at larger scale                                                           │
│                                                                                                                 │
│  Your job when using this skill is to figure out where the user is in this process and then jump in and help    │
│  them progress through these stages. So for instance, maybe they're like "I want to make a skill for X". You    │
│  can help narrow down what they mean, write a draft, write the test cases, figure out how they want to          │
│  evaluate, run all the prompts, and repeat.                                                                     │
│                                                                                                                 │
│  On the other hand, maybe th                                                                                    │
│                                                                                                                 │
│  …[truncated]                                                                                                   │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [54]:
# Build an agent with skill-creator + the existing library, then ask it to author a NEW skill
creator_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You author new agent skills. When asked to create a skill, use the skill-creator skill — "
        "follow its SOP exactly, validate the frontmatter, and write the final SKILL.md to disk."
    ),
    tools=[],
    backend=BACKEND,
    skills=["/skills"],
)

print("Running skill-creator agent...\n")
result_creator = creator_agent.invoke({
    "messages": [{"role": "user", "content": (
        "Create a new skill called 'daily-standup'. Its job: given a list of yesterday's work, "
        "today's plan, and blockers, produce a clean markdown standup post suitable for Slack. "
        "Save the new skill at /skills/daily-standup/SKILL.md and return the path."
    )}]
}, config={"recursion_limit": 30})


Running skill-creator agent...



In [55]:
# Did it author a new skill?
from pathlib import Path
new_skill_path = Path("./skills/daily-standup/SKILL.md")
if new_skill_path.exists():
    print(f"✅ Agent wrote a new skill at {new_skill_path}\n")
    pretty_skill(str(new_skill_path))
else:
    print("⚠️ daily-standup skill not on disk — inspecting state files...")
    state_files = result_creator.get("files", {})
    print(f"Files in state: {list(state_files.keys())}")


✅ Agent wrote a new skill at skills/daily-standup/SKILL.md



╭──────────────────────────────── 📄 skills/daily-standup/SKILL.md — frontmatter ─────────────────────────────────╮
│ name: daily-standup                                                                                             │
│ description: Generate a clean markdown standup post for Slack from yesterday's work, today's plan, and          │
│ blockers. Use this skill whenever the user wants to write a daily standup, daily sync, or team update — even if │
│ they don't explicitly say "skill". Triggers on phrases like "standup", "daily update", "what I did yesterday",  │
│ "today's plan", "blockers", "sync post", or when asked to format work updates for Slack.                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────── 📄 skills/daily-standup/SKILL.md — body ────────────────────────────────────╮
│                                                                                                                 │
│                                                  Daily Standup                                                  │
│                                                                                                                 │
│  Transform raw standup notes into a polished Slack-ready markdown post.                                         │
│                                                                                                                 │
│  Input                                                                                                          │
│                                                                                                                 │
│  Collect these three pieces of information from the user:                                                       │
│                                                                                                                 │
│   • Yesterday: What was accomplished                                                                            │
│   • Today: What will be worked on                                                                               │
│   • Blockers: Any impediments (or "None" / "None!" if clear)                                                    │
│                                                                                                                 │
│  Output Format                                                                                                  │
│                                                                                                                 │
│  Always produce Slack-formatted markdown using this structure:                                                  │
│                                                                                                                 │
│                                                                                                                 │
│   *Daily Standup* 📅 <date>                                                                                     │
│                                                                                                                 │
│   *Yesterday ✅*                                                                                                │
│   • [yesterday's work item 1]                                                                                   │
│   • [yesterday's work item 2]                                                                                   │
│   • ...                                                                                                         │
│                                                                                                                 │
│   *Today 🎯*                                                                                                    │
│   • [today's plan item 1]                                                                                       │
│   • [today's plan item 2]                                                                                       │
│   • ...                                                                                                         │
│                                                                                                                 │
│   *Blockers 🚧*                                                                                                 │
│   • [blocker 1]                                                                                                 │
│   • [blocker 2]                                                                                                 │
│   • ...                                                    

**What just happened:** the agent loaded the `skill-creator` skill's SOP (which says how to structure a SKILL.md, what frontmatter fields matter, how to write a good `description`), followed that procedure, and produced a fresh `daily-standup` skill — ready to load on the next `create_deep_agent` call. **A skill writing a skill** is the recursive-improvement loop that makes the Skills pattern compound.


---
## Section 6.5 — Wiring up an MCP Server

Skills package *procedures*. **MCP servers** package *tools* — services that live outside the agent process and expose typed actions over a protocol. Most production deep agents use both: a SKILL.md tells the agent *what to do*, and one or more MCP servers give it *the actions to take*.

Below we wire up a tiny stdio MCP server (`mcp_demo_server.py`, shipped alongside this notebook) that exposes two business tools — `get_team_oncall` and `get_company_metrics` — then hand those tools to a deep agent. The agent calls them just like any other tool, but the implementation lives in a separate process. Swap in a real Postgres / Slack / Linear MCP server and the pattern is identical.


In [56]:
# Quick peek at the MCP server we're about to connect to
with open('mcp_demo_server.py') as f:
    server_src = f.read()
from rich.syntax import Syntax
_console.print(Panel(Syntax(server_src, 'python', theme='monokai', line_numbers=True, word_wrap=False),
                     title='📡 mcp_demo_server.py', border_style='magenta'))


╭───────────────────────────────────────────── 📡 mcp_demo_server.py ─────────────────────────────────────────────╮
│    1 """Minimal stdio MCP server for the Session 4 demo.                                                        │
│    2                                                                                                            │
│    3 Exposes two business-relevant tools so the agent has something concrete to call                            │
│    4 through the MCP protocol. Run via:                                                                         │
│    5                                                                                                            │
│    6     python mcp_demo_server.py        # stdio (the notebook launches it this way)                           │
│    7 """                                                                                                        │
│    8                                                                                                            │
│    9 from datetime import date                                                                                  │
│   10                                                                                                            │
│   11 from mcp.server.fastmcp import FastMCP                                                                     │
│   12                                                                                                            │
│   13 mcp = FastMCP("session4-demo")                                                                             │
│   14                                                                                                            │
│   15                                                                                                            │
│   16 @mcp.tool()                                                                                                │
│   17 def get_team_oncall(date_iso: str | None = None) -> dict:                                                  │
│   18     """Return who's on call for a given date (YYYY-MM-DD). Defaults to today.                              │
│   19                                                                                                            │
│   20     Returns the primary and secondary on-call engineers plus their pager handles.                          │
│   21     """                                                                                                    │
│   22     rota = {                                                                                               │
│   23         "2026-05-12": {"primary": "Asha P.", "secondary": "Marco L.", "pager": "+1-415-555-0142"},         │
│   24         "2026-05-13": {"primary": "Jordan W.", "secondary": "Priya S.", "pager": "+1-415-555-0143"},       │
│   25         "2026-05-14": {"primary": "Mei C.", "secondary": "Tomás R.", "pager": "+1-415-555-0144"},          │
│   26     }                                                                                                      │
│   27     d = date_iso or date.today().isoformat()                                                               │
│   28     return {"date": d, **rota.get(d, {"primary": "TBD", "secondary": "TBD", "pager": "-"})}                │
│ [

In [57]:
# Connect to the MCP server via stdio — same pattern works for any MCP server
# (filesystem, postgres, slack, github, etc — just swap the command/args/transport).
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient({
    "demo": {
        "command": "uv",
        "args": ["run", "python", "mcp_demo_server.py"],
        "transport": "stdio",
    }
})

mcp_tools = await mcp_client.get_tools()

print(f"Connected. {len(mcp_tools)} MCP tools available:")
for t in mcp_tools:
    print(f"  • {t.name}: {t.description.splitlines()[0]}")


Connected. 2 MCP tools available:
  • get_team_oncall: Return who's on call for a given date (YYYY-MM-DD). Defaults to today.
  • get_company_metrics: Return high-level company metrics for a given quarter.


In [58]:
# Hand the MCP tools to a deep agent. They look like any other LangChain tool.
mcp_agent = create_deep_agent(
    model=MODEL,
    system_prompt=(
        "You answer questions about the company using the available MCP tools. "
        "Always cite which tool you called."
    ),
    tools=mcp_tools,
    backend=BACKEND,
)

print("Asking the agent two questions that should fire different MCP tools...\n")
result_mcp = await mcp_agent.ainvoke({
    "messages": [{"role": "user", "content": (
        "Who is on call for 2026-05-13, and what was our Q1-2026 ARR? "
        "One sentence per answer, with the tool name in parentheses."
    )}]
}, config={"recursion_limit": 10})

print("FINAL:", str(result_mcp["messages"][-1].content)[:600])


Asking the agent two questions that should fire different MCP tools...

FINAL: 

Jordan W. is primary on call for 2026-05-13, with Priya S. as secondary (get_team_oncall). Our Q1-2026 ARR was $21.7 million (get_company_metrics).


In [59]:
# Trace which MCP tools fired
format_messages(result_mcp["messages"])


╭─────────────────────────────────────────────────── 🧑 Human ────────────────────────────────────────────────────╮
│ Who is on call for 2026-05-13, and what was our Q1-2026 ARR? One sentence per answer, with the tool name in     │
│ parentheses.                                                                                                    │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ 🔧 Tool Call: get_team_oncall                                                                                   │
│    Args: {                                                                                                      │
│   "date_iso": "2026-05-13"                                                                                      │
│ }                                                                                                               │
│    ID: call_5df97dc90aca4a089c                                                                                  │
│                                                                                                                 │
│ 🔧 Tool Call: get_company_metrics                                                                               │
│    Args: {                                                                                                      │
│   "quarter": "Q1-2026"                                                                                          │
│ }                                                                                                               │
│    ID: call_7c53e2372aa84dbbbb                                                                                  │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "date": "2026-05-13",                                                                                         │
│   "primary": "Jordan W.",                                                                                       │
│   "secondary": "Priya S.",                                                                                      │
│   "pager": "+1-415-555-0143"                                                                                    │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── 🔧 Tool Output ─────────────────────────────────────────────────╮
│ {                                                                                                               │
│   "quarter": "Q1-2026",                                                                                         │
│   "arr_usd": 21700000,                                                                                          │
│   "customers": 468,                                                                                             │
│   "nps": 51,                                                                                                    │
│   "uptime_pct": 99.94                                                                                           │
│ }                                                                                                               │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭───────────────────────────────────────────────────── 📝 AI ─────────────────────────────────────────────────────╮
│                                                                                                                 │
│                                                                                                                 │
│ Jordan W. is primary on call for 2026-05-13, with Priya S. as secondary (get_team_oncall). Our Q1-2026 ARR was  │
│ $21.7 million (get_company_metrics).                                                                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

**What we just demonstrated:**
1. A tiny FastMCP server exposes 2 typed tools over **stdio transport** — could equally be Streamable HTTP, SSE, or WebSocket.
2. `MultiServerMCPClient` wraps the stdio process and surfaces the tools as standard LangChain `BaseTool` objects.
3. `create_deep_agent(tools=mcp_tools)` accepts them with no special handling — to the agent loop they're indistinguishable from in-process Python tools.
4. The agent picks the right tool per question and cites it.

**In production, the pattern is identical** — swap our toy server for any of the 5,800+ community MCP servers (Postgres, GitHub, Slack, Linear, Notion, Sentry, the LangSmith Agent Server endpoint, your internal API…). Skills + MCP + subagents are the three composition primitives every production deep agent uses.


---
## Section 7: Key Takeaways

1. **Agent Skills are packaged procedures** — `SKILL.md` in a folder, loaded on demand. Originated at Anthropic, now an open framework-agnostic standard adopted by OpenAI, Google, Cursor, GitHub, and deepagents.
2. **Progressive disclosure** keeps the system prompt small — Tier 1 (description) on startup, Tier 2 (body) on match, Tier 3 (bundled files) when referenced. Scripts execute without entering context.
3. **Skills compose with everything** — MCP tools (via `allowed-tools`), custom Python tools, and subagents (via per-subagent `skills=[...]`).
4. **Authoring is cheap** — a useful skill is often 30 lines of markdown. No code.
5. **One parameter in deepagents** — `create_deep_agent(skills=["./skills"])`. SkillsMiddleware handles the rest.

---
## Exercises

### Exercise 1: A code-review skill
Author `./skills/code-review/SKILL.md` that walks the agent through running `ruff check` (use a `Bash` tool) on a target file and reporting findings as a markdown summary. Frontmatter description should trigger on phrases like "review my code" or "check this file for issues".

### Exercise 2: A skill for *your* workflow
Pick one repetitive procedure from your own work. Author it as a skill in ~10 lines. Run it against an agent. See how it feels.

### Exercise 3: Stretch — context savings under composition
Build an agent with three subagents, each scoped to two skills. Run a multi-deliverable task. Measure the supervisor's final message count vs the total of all subagent files. Plot the result. (Use the telemetry pattern from Session 3.)

---
## Next Session: May 27 — Multi-Agent Workflows

Sessions 1–3 made the agent **think and remember**. Today (Session 4) made it **capable** with Skills + MCP. **Next session (May 27)** we make it **scale** — supervisor patterns, handoffs, parallel execution, and assembling the full Deep Research Agent the series has been building toward. Session 6 (Eval & Production) closes the series.